In [ ]:
"""
Sales Forecasting Model: Revenue & COGS from Date Only
"""
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from sklearn.ensemble import GradientBoostingRegressor
from sklearn.metrics import mean_absolute_percentage_error
from sklearn.model_selection import TimeSeriesSplit
import shap

# ── 1. Feature Engineering ────────────────────────────────────────────────────
def add_features(df: pd.DataFrame) -> pd.DataFrame:
    """Extract rich date-based features, including Fourier seasonal terms."""
    df = df.copy()
    df['year']      = df['Date'].dt.year
    df['month']     = df['Date'].dt.month
    df['day']       = df['Date'].dt.day
    df['dow']       = df['Date'].dt.dayofweek      # 0=Mon, 6=Sun
    df['doy']       = df['Date'].dt.dayofyear
    df['quarter']   = df['Date'].dt.quarter
    df['week']      = df['Date'].dt.isocalendar().week.astype(int)
    df['is_weekend']= (df['dow'] >= 5).astype(int)

    # Linear time trend
    df['t']         = (df['Date'] - pd.Timestamp('2012-07-04')).dt.days
    df['year_norm'] = df['year'] - 2012
    df['year_sq']   = df['year_norm'] ** 2

    # Fourier (cyclic) seasonal features to capture non-linear seasonality
    for k in [1, 2, 3, 4]:
        df[f'month_sin_{k}'] = np.sin(2*np.pi*k*df['month']/12)
        df[f'month_cos_{k}'] = np.cos(2*np.pi*k*df['month']/12)
    for k in [1, 2, 3]:
        df[f'doy_sin_{k}'] = np.sin(2*np.pi*k*df['doy']/365)
        df[f'doy_cos_{k}'] = np.cos(2*np.pi*k*df['doy']/365)
    for k in[1, 2]:
        df[f'dow_sin_{k}'] = np.sin(2*np.pi*k*df['dow']/7)
        df[f'dow_cos_{k}'] = np.cos(2*np.pi*k*df['dow']/7)
        df[f'week_sin_{k}'] = np.sin(2*np.pi*k*df['week']/52)
        df[f'week_cos_{k}'] = np.cos(2*np.pi*k*df['week']/52)

    return df

FEATURE_COLS =[
    'year', 'month', 'day', 'dow', 'doy', 'quarter', 'week', 'is_weekend',
    't', 'year_norm', 'year_sq',
    *[f'month_{t}_{k}' for t in ['sin','cos'] for k in[1,2,3,4]],
    *[f'doy_{t}_{k}'   for t in ['sin','cos'] for k in [1,2,3]],
    *[f'dow_{t}_{k}'   for t in['sin','cos'] for k in [1,2]],
    *[f'week_{t}_{k}'  for t in ['sin','cos'] for k in [1,2]],
]

GBM_PARAMS = dict(
    n_estimators=800, max_depth=4, learning_rate=0.05,
    subsample=0.75, min_samples_leaf=5, random_state=42
)

# ── 2. Explain Model (SHAP & Feature Importance) ──────────────────────────────
def explain_model(model, X_train, target_name):
    """Generates and saves SHAP values and Feature Importance plots."""
    print(f"\n--- Generating Explanations for {target_name} ---")
    
    # Built-in Feature Importance
    importances = model.feature_importances_
    indices = np.argsort(importances)[-15:] # Top 15 features
    
    plt.figure(figsize=(10, 6))
    plt.title(f'Top 15 Feature Importances (GBM) - {target_name}')
    plt.barh(range(len(indices)), importances[indices], align='center')
    plt.yticks(range(len(indices)), [FEATURE_COLS[i] for i in indices])
    plt.xlabel('Relative Importance')
    plt.tight_layout()
    plt.savefig(f'feature_importance_{target_name}.png')
    plt.close()
    
    # SHAP Values
    # Note: Using a sample if data is too large to speed up SHAP calculation
    X_sample = shap.utils.sample(X_train, 1000) if len(X_train) > 1000 else X_train
    explainer = shap.TreeExplainer(model)
    shap_values = explainer.shap_values(X_sample)
    
    plt.figure(figsize=(10, 6))
    shap.summary_plot(shap_values, X_sample, show=False)
    plt.title(f'SHAP Summary Plot - {target_name}')
    plt.tight_layout()
    plt.savefig(f'shap_summary_{target_name}.png')
    plt.close()
    print(f"Saved plots: feature_importance_{target_name}.png, shap_summary_{target_name}.png")

# ── 3. Train, Cross-Validate & Predict ────────────────────────────────────────
def train_predict(sales_path: str, output_path: str,
                  forecast_start='2023-01-01', forecast_end='2024-07-01'):
    
    # Load & engineer features
    sales = pd.read_csv(sales_path, parse_dates=['Date'])
    sales = sales.sort_values('Date').reset_index(drop=True)
    sales = sales[sales['Date'].dt.year >= 2013].reset_index(drop=True) # Start from 2013
    
    sales_f = add_features(sales)
    
    X_full = sales_f[FEATURE_COLS]
    
    # Xử lý Leakage & Cross-validation bằng TimeSeriesSplit
    # TimeSeriesSplit tránh rò rỉ dữ liệu tương lai vào quá khứ
    tscv = TimeSeriesSplit(n_splits=3)
    
    print("--- Time Series Cross-Validation ---")
    for target in ['Revenue', 'COGS']:
        y_full = np.log(sales[target])
        fold_mapes =[]
        
        for fold, (train_index, val_index) in enumerate(tscv.split(X_full)):
            X_train_cv, X_val_cv = X_full.iloc[train_index], X_full.iloc[val_index]
            y_train_cv, y_val_cv = y_full.iloc[train_index], y_full.iloc[val_index]
            
            m_cv = GradientBoostingRegressor(**GBM_PARAMS).fit(X_train_cv, y_train_cv)
            pred_val = np.exp(m_cv.predict(X_val_cv))
            actual_val = np.exp(y_val_cv)
            
            mape = mean_absolute_percentage_error(actual_val, pred_val) * 100
            fold_mapes.append(mape)
            
        print(f"  {target} CV MAPEs across folds: {[f'{m:.2f}%' for m in fold_mapes]}")
        print(f"  {target} Average CV MAPE: {np.mean(fold_mapes):.2f}%")

    # Full retrain + forecast + Explanation
    print("\n--- Full Retrain & Forecasting ---")
    test_dates = pd.date_range(forecast_start, forecast_end)
    test_df = pd.DataFrame({'Date': test_dates})
    test_f  = add_features(test_df)
    X_test  = test_f[FEATURE_COLS]

    sub = test_df.copy()
    for target in ['Revenue', 'COGS']:
        y_full = np.log(sales[target])
        final_model = GradientBoostingRegressor(**GBM_PARAMS).fit(X_full, y_full)
        
        # Explain the final model
        explain_model(final_model, X_full, target)
        
        # Predict
        sub[target] = np.exp(final_model.predict(X_test)).round(2)

    sub['Date'] = sub['Date'].dt.strftime('%Y-%m-%d')
    sub.to_csv(output_path, index=False)
    print(f"\n  Saved {len(sub)} rows to {output_path}")
    return sub

if __name__ == '__main__':
    # Lưu ý đổi đường dẫn sales_path thành file thực tế của bạn nếu chạy local
    sub = train_predict(
        sales_path='/kaggle/input/competitions/datathon-2026-round-1/sales.csv',
        output_path='submission.csv'
    )
    print(sub.head())

--- Time Series Cross-Validation ---
  Revenue CV MAPEs across folds: ['23.17%', '45.58%', '22.75%']
  Revenue Average CV MAPE: 30.50%
  COGS CV MAPEs across folds: ['19.85%', '42.80%', '22.17%']
  COGS Average CV MAPE: 28.27%

--- Full Retrain & Forecasting ---

--- Generating Explanations for Revenue ---
